# Seq2Seq (encoder-decoder LSTM) - SAFEX white maize, 5-day-ahead forecast

An **encoder** LSTM reads the last `ENC_LEN` business days of *price + ZAR/USD + rainfall + fuel price* and
compresses them into a state vector. A **decoder** LSTM starts from that state and writes the next
`PRED_STEPS` (5 = one trading week) prices, one day at a time.

The architecture, teacher forcing during training and the step-by-step inference loop are the same as the
soybean notebook. Two things were redesigned:

* **Many training windows instead of one.** The original trained on a single sample (the whole history as
  one encoder input, 500 epochs, batch size 1). Here every position in the training period gives one
  (history -> next 5 days) sample, so the network sees hundreds of examples.
* **The external drivers enter the encoder** as extra input channels. The decoder only sees the
  previous price, so no future driver values are required - the forecast is a genuine forecast.

Price scaling is kept from the original: `log1p`, then subtract the encoder window's mean, so the network
works with *relative* price movements. That removes the "prices outgrew the training range" problem.

## 1. Imports

In [ ]:
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
tf.keras.utils.set_random_seed(42)
%matplotlib inline

## 2. Load and align the datasets

In [ ]:
# ---- Data settings ---------------------------------------------------------
PRICES_FILE = 'safex_prices.csv'   # Date, Close (R/ton)  [+ optional Open, High, Low, Volume]
EXOG_FILE   = 'sa_factors.csv'     # Date, ZAR_USD, Rainfall_mm, Fuel_Price
TARGET_COL  = 'Close'              # SAFEX white maize price column
EXOG_COLS   = ['ZAR_USD', 'Rainfall_mm', 'Fuel_Price']   # external drivers
TRAIN_FRAC  = 0.8                  # chronological split: first 80% train, last 20% test

# ---- Seq2Seq settings -----------------------------------------------------------
ENC_LEN    = 60      # business days of history given to the encoder
PRED_STEPS = 5       # business days to forecast (5 = one trading week)
LATENT_DIM = 32      # LSTM hidden units
DROPOUT    = 0.20
EPOCHS     = 200     # upper limit - early stopping usually ends training sooner
BATCH_SIZE = 32

In [ ]:
# Load the SAFEX white maize prices and the South African external drivers.
# (If your dates are written dd/mm/yyyy, add dayfirst=True to both read_csv calls.)
prices = pd.read_csv(PRICES_FILE, parse_dates=['Date'], index_col='Date')
exog_data = pd.read_csv(EXOG_FILE, parse_dates=['Date'], index_col='Date')

# Tidy up: ascending dates and no duplicated dates
prices = prices.sort_index()
prices = prices[~prices.index.duplicated(keep='last')]
exog_data = exog_data.sort_index()
exog_data = exog_data[~exog_data.index.duplicated(keep='last')]

# Fail early, with a readable message, if a column name does not match the settings above
missing = [c for c in [TARGET_COL] if c not in prices.columns] + \
          [c for c in EXOG_COLS if c not in exog_data.columns]
if missing:
    raise KeyError(f'Column(s) not found in the CSV files: {missing}')

In [ ]:
# Safeguard: put lower-frequency drivers (e.g. monthly fuel prices) on a daily calendar BEFORE the
# join, carrying the last known value forward. Otherwise a monthly series only matches ~12 dates a
# year, and the inner join below would silently throw away almost all of the daily price rows.
# (.last().ffill() also fills blanks if all drivers sit in one wide file with empty cells.)
exog_data = exog_data.resample('D').last().ffill()

# Merge both dataframes with an inner join so that the dates align
data = prices.join(exog_data, how='inner')

# Business-day calendar; forward-fill weekend / holiday gaps
data = data.resample('B').ffill()
data = data.dropna(subset=[TARGET_COL] + EXOG_COLS)

In [ ]:
print(f'{len(data)} business days   {data.index.min().date()} -> {data.index.max().date()}')
data[[TARGET_COL] + EXOG_COLS].describe().round(2)

In [ ]:
data[TARGET_COL].plot(figsize=(12, 4), title='SAFEX white maize (R/ton)')
plt.show()

## 3. Train / test partitioning

Same chronological 80 / 20 split as the other notebooks. A *forecast origin* `t` is the first day to be
predicted: the encoder sees days `t - ENC_LEN ... t - 1`, the decoder predicts days `t ... t + PRED_STEPS - 1`.

* **Training origins** - the whole 5-day target stays inside the training period
* **Test origins** - the first predicted day is in the test period (the encoder may still look back into
  the training period; that history is known at forecast time)

In [ ]:
n = len(data)
train_size = int(n * TRAIN_FRAC)

train_origins = range(ENC_LEN, train_size - PRED_STEPS + 1)
test_origins = range(train_size, n - PRED_STEPS + 1)

dates = data.index
print('Training origins:', len(train_origins), f'({dates[train_origins[0]].date()} -> {dates[train_origins[-1]].date()})')
print('Test origins    :', len(test_origins), f'({dates[test_origins[0]].date()} -> {dates[test_origins[-1]].date()})')
print('Encoding interval:', ENC_LEN, 'business days | Prediction interval:', PRED_STEPS, 'business days')

## 4. Transform the series

* Price: `log1p`, then subtract the **mean of the encoder window** (so the network sees relative moves).
* Drivers: standardised with **training-period** statistics only.

In [ ]:
close = data[TARGET_COL].to_numpy(dtype='float64')
log_close = np.log1p(close)

exog_scaler = StandardScaler().fit(data[EXOG_COLS].iloc[:train_size])
exog_scaled = exog_scaler.transform(data[EXOG_COLS])            # (n, number of drivers)

n_features = 1 + len(EXOG_COLS)


def encode_window(t):
    """Encoder input for forecast origin t: (ENC_LEN, n_features), the window mean, last relative price."""
    enc_close = log_close[t - ENC_LEN:t]
    mean = enc_close.mean()
    feats = np.column_stack([enc_close - mean, exog_scaled[t - ENC_LEN:t]])
    return feats, mean, enc_close[-1] - mean


def build_windows(origins):
    enc_x, dec_in, dec_y, means = [], [], [], []
    for t in origins:
        feats, mean, last_rel = encode_window(t)
        target = log_close[t:t + PRED_STEPS] - mean                # relative log price of the next days
        enc_x.append(feats)
        dec_y.append(target)
        dec_in.append(np.concatenate([[last_rel], target[:-1]]))   # teacher forcing: lagged target series
        means.append(mean)
    return (np.array(enc_x),
            np.array(dec_in)[:, :, None],
            np.array(dec_y)[:, :, None],
            np.array(means))


enc_tr, dec_in_tr, dec_y_tr, mean_tr = build_windows(train_origins)
enc_te, dec_in_te, dec_y_te, mean_te = build_windows(test_origins)

print('encoder input :', enc_tr.shape, '(samples, ENC_LEN, price + drivers)')
print('decoder input :', dec_in_tr.shape)
print('decoder target:', dec_y_tr.shape)

## 5. Model building

In [ ]:
# Define an input series and encode it with an LSTM
encoder_inputs = Input(shape=(None, n_features))
encoder = LSTM(LATENT_DIM, dropout=DROPOUT, return_state=True)
_, state_h, state_c = encoder(encoder_inputs)

# Only the final states are kept: they are the "context" vector used to start the decoder
encoder_states = [state_h, state_c]

# Decoder: starts from the encoder states; teacher-forcing inputs (the lagged target) are fed in here
decoder_inputs = Input(shape=(None, 1))
decoder_lstm = LSTM(LATENT_DIM, dropout=DROPOUT, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(decoder_inputs, initial_state=encoder_states)

decoder_dense = Dense(1)                      # one continuous output per time step
decoder_outputs = decoder_dense(decoder_outputs)

# Turns `encoder_input_data` & `decoder_input_data` into `decoder_target_data`
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.summary()

In [ ]:
model.compile(optimizer=Adam(learning_rate=1e-3), loss='mean_absolute_error')

early_stop = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)
history = model.fit([enc_tr, dec_in_tr], dec_y_tr,
                    batch_size=BATCH_SIZE, epochs=EPOCHS,
                    validation_split=0.1,          # the LAST 10% of the training windows
                    callbacks=[early_stop], verbose=2)

In [ ]:
plt.plot(history.history['loss'], label='Train')
plt.plot(history.history['val_loss'], label='Valid')
plt.xlabel('Epoch')
plt.ylabel('Mean Absolute Error Loss')
plt.title('Loss over time')
plt.legend()
plt.show()

## 6. Inference models

Same pattern as before: an encoder model maps the history to state vectors, and a one-step decoder model is
called in a loop, feeding each prediction back in as the next input. The loop is **batched** (all windows
at once), so evaluating hundreds of test windows takes `PRED_STEPS` model calls instead of hundreds.

In [ ]:
# Encoder model: history -> state vectors
encoder_model = Model(encoder_inputs, encoder_states)

# One-step decoder: (previous value, states) -> (next value, new states)
dec_state_h = Input(shape=(LATENT_DIM,))
dec_state_c = Input(shape=(LATENT_DIM,))
dec_step_input = Input(shape=(None, 1))
dec_step_out, new_h, new_c = decoder_lstm(dec_step_input, initial_state=[dec_state_h, dec_state_c])
dec_step_out = decoder_dense(dec_step_out)
decoder_model = Model([dec_step_input, dec_state_h, dec_state_c], [dec_step_out, new_h, new_c])


def decode_batch(enc_x):
    """enc_x: (batch, ENC_LEN, n_features) -> predicted relative log prices (batch, PRED_STEPS)."""
    states = encoder_model.predict(enc_x, verbose=0)
    target_seq = enc_x[:, -1, 0].reshape(-1, 1, 1)             # start from the last observed (relative) price
    decoded = np.zeros((enc_x.shape[0], PRED_STEPS))

    for i in range(PRED_STEPS):
        output, h, c = decoder_model.predict([target_seq] + list(states), verbose=0)
        decoded[:, i] = output[:, 0, 0]
        target_seq = output                                     # feed the prediction back in
        states = [h, c]
    return decoded

## 7. Evaluate on the test period

Predictions are converted back to R/ton (`expm1`, adding the window mean back) and compared with the naive
benchmark "the price stays at the last observed value", **by forecast horizon**. Errors normally grow with
the horizon.

In [ ]:
pred_rel = decode_batch(enc_te)                                          # (windows, PRED_STEPS)
pred_price = np.expm1(pred_rel + mean_te[:, None])
actual_price = np.expm1(dec_y_te[:, :, 0] + mean_te[:, None])
last_price = np.expm1(enc_te[:, -1, 0] + mean_te)                        # last observed price per window
naive_price = np.repeat(last_price[:, None], PRED_STEPS, axis=1)

horizon_labels = [f'day +{h}' for h in range(1, PRED_STEPS + 1)]
table = pd.DataFrame({
    'Seq2Seq MAE': np.abs(pred_price - actual_price).mean(axis=0),
    'Naive MAE': np.abs(naive_price - actual_price).mean(axis=0),
    'Seq2Seq MAPE %': (np.abs(pred_price - actual_price) / actual_price).mean(axis=0) * 100,
}, index=horizon_labels)
table.loc['all days'] = [np.abs(pred_price - actual_price).mean(),
                         np.abs(naive_price - actual_price).mean(),
                         (np.abs(pred_price - actual_price) / actual_price).mean() * 100]
table.round(2)

In [ ]:
# One example: the last test window
i = len(test_origins) - 1
t = test_origins[i]
tail = 50

plt.figure(figsize=(10, 5))
plt.plot(dates[t - tail:t], close[t - tail:t], label='Encoding series (actual)')
plt.plot(dates[t:t + PRED_STEPS], actual_price[i], color='orange', marker='o', label='Target series (actual)')
plt.plot(dates[t:t + PRED_STEPS], pred_price[i], color='teal', linestyle='--', marker='o', label='Predictions')
plt.title(f'Last {tail} days of history, the next {PRED_STEPS} days, and the forecast')
plt.ylabel('R / ton')
plt.legend()
plt.show()

In [ ]:
# Day-ahead (+1) and week-ahead (+5) forecasts across the whole test period
plt.figure(figsize=(12, 5))
plt.plot(dates[list(test_origins)], actual_price[:, 0], color='black', linewidth=1, label='actual (day +1)')
plt.plot(dates[list(test_origins)], pred_price[:, 0], label='Seq2Seq, day +1')
plt.plot(dates[[t + PRED_STEPS - 1 for t in test_origins]], pred_price[:, -1],
         label=f'Seq2Seq, day +{PRED_STEPS}', alpha=0.8)
plt.title('Seq2Seq forecasts across the test period')
plt.ylabel('R / ton')
plt.legend()
plt.show()

## 8. Forecast the next trading week

The encoder reads the latest `ENC_LEN` days; no future driver values are needed.

In [ ]:
feats_next, mean_next, _ = encode_window(len(data))
next_rel = decode_batch(feats_next[None, :, :])[0]
next_prices = np.expm1(next_rel + mean_next)

next_dates = pd.bdate_range(data.index[-1] + pd.offsets.BDay(1), periods=PRED_STEPS)
pd.DataFrame({'forecast (R/ton)': next_prices.round(2)}, index=next_dates)

The original notebook flipped the sign of the decoded values (`a = -a`) as a workaround for its
single-window transform. That is no longer needed: the decode is simply `expm1(prediction + window mean)`.